# 03 · Partición interna entrenamiento / validación

Divide el 80 % de entrenamiento en un **entrenamiento interno** (80 %) y una **validación** (20 %), de forma estratificada. Los generadores y todos los clasificadores usan solo el entrenamiento interno; la validación se usa únicamente para elegir hiperparámetros (notebooks 04 a 07).

> Ejecutar una sola vez, inmediatamente después del notebook 01: este paso sobrescribe `outputs/splits/X_train_raw_in.csv` con el subconjunto interno.

In [ ]:
import json
import numpy as np
import pandas as pd
from collections import Counter
from sklearn.model_selection import train_test_split

import os
import sys

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
sys.path.append(PROJECT_ROOT)
from utils.config_loader import load_config

config = load_config("dataset_config.yaml")
RANDOM_STATE = config.get("random_state", 42)
print("Dataset:", config["dataset_name"])
VAL_RATIO = 0.2

## Carga y partición

In [ ]:
X_train_proc = pd.read_csv("outputs/data/X_train.csv")
y_train = pd.read_csv("outputs/data/y_train.csv")["y"].values
X_train_raw = pd.read_csv("outputs/splits/X_train_raw_in.csv")

# La partición se hace sobre índices para que la versión escalada y la original coincidan fila a fila
indices = np.arange(len(X_train_proc))
idx_train, idx_val = train_test_split(indices, test_size=VAL_RATIO, stratify=y_train, random_state=RANDOM_STATE)

X_train_in_proc, X_val_in_proc = X_train_proc.iloc[idx_train], X_train_proc.iloc[idx_val]
y_train_in, y_val_in = y_train[idx_train], y_train[idx_val]
X_train_in_raw = X_train_raw.iloc[idx_train]

counter = Counter(y_train_in)
samples_to_generate = max(counter.values()) - min(counter.values())
print(f"Entrenamiento interno: { {int(k): int(v) for k, v in counter.items()} } | registros a generar para 1:1: {samples_to_generate}")

## Guardado

In [ ]:
OUTPUT_DIR = "outputs/splits"
X_train_in_proc.to_csv(f"{OUTPUT_DIR}/X_train_in.csv", index=False)
X_val_in_proc.to_csv(f"{OUTPUT_DIR}/X_val_in.csv", index=False)
pd.DataFrame({"y": y_train_in}).to_csv(f"{OUTPUT_DIR}/y_train_in.csv", index=False)
pd.DataFrame({"y": y_val_in}).to_csv(f"{OUTPUT_DIR}/y_val_in.csv", index=False)
X_train_in_raw.to_csv(f"{OUTPUT_DIR}/X_train_raw_in.csv", index=False)

with open(f"{OUTPUT_DIR}/balance_config.json", "w") as f:
    json.dump({"target_per_class": int(max(counter.values())),
               "samples_to_generate": int(samples_to_generate),
               "minority_label": int(min(counter, key=counter.get))}, f, indent=4)
print("Partición interna guardada.")